# FSCT 8561 — Lab 8: Image Steganography and LSB Forensics
**Security Applications | Guided Jupyter Laboratory | 3 hours**  
**Instructor:** Dr. Maryam R. Aliabadi  
**Reading:** *Mastering Python for Networking and Security*, Chapter 14; Pillow documentation; Stepic project page.

### Scenario
You are a security analyst examining a **fictional insider-threat simulation**. The team has supplied benign image carriers and invented data (a private-network target label and a list of service ports). Your goal is to understand how least-significant-bit (LSB) image steganography works, build and verify a small encoder/decoder, and document what a defender can and cannot infer.

**Safety and scope:** Work only with the supplied images and fictional lab data. Do not scan external services, send stego files to third parties, or treat an embedded string as proof of actual exfiltration.

### Learning outcomes
1. Explain pixels, RGB/RGBA channels, binary representations, and LSB changes.
2. Modify and read LSBs in Python with small worked examples.
3. Encode and recover a short text message in a PNG using the supplied algorithm as a reference.
4. Use Stepic to encode and decode a separate message, then compare both approaches.
5. Calculate capacity and report image differences, sizes, and limitations.
6. Present reproducible evidence, error handling, and a security reflection.



## Part 0 — Setup and evidence inventory
Extract `Lab8.zip`. Put this notebook in the same directory as the extracted `Lab8/` folder. The supplied archive contains `compay_logo.png`, `profile.png`, `python.png`, `python_secrets.png`, and `steganography_LSB.py`.

Install Pillow and Stepic if necessary. In Jupyter, run the following installation cell once; restart the kernel if prompted.

In [ ]:
# Uncomment if needed, then run once:
# %pip install pillow stepic

from pathlib import Path
from PIL import Image
import hashlib

ASSET_DIR = Path("Lab8")
LOGO = ASSET_DIR / "compay_logo.png"
PROFILE = ASSET_DIR / "profile.png"
PYTHON_IMG = ASSET_DIR / "python.png"
OUTPUT_DIR = Path("Lab8_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

for path in [LOGO, PROFILE, PYTHON_IMG]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Check where you extracted Lab8.zip")
    with Image.open(path) as image:
        print(path.name, image.size, image.mode, path.stat().st_size, "bytes")

**Check-in 0:** What does `RGBA` mean? Why might a forensic analyst calculate a file's SHA-256 hash?

**Evidence workflow:** The hashes recorded below are **baseline values**. You will reuse the same function in Part 5 to compare the original images with their stego-images, then verify that message extraction does not change a stego-image. A hash difference proves the file bytes differ; it does **not** prove steganography.

In [ ]:
def sha256_file(path):
    """Calculate the SHA-256 digest of a file's exact bytes."""
    with open(path, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()

# Save baseline hashes now; we will use these values again in Part 5.
original_logo_hash = sha256_file(LOGO)
original_profile_hash = sha256_file(PROFILE)

print("Original logo SHA-256:   ", original_logo_hash)
print("Original profile SHA-256:", original_profile_hash)


## Part 1 — Binary and LSB fundamentals 
An 8-bit channel value ranges from 0 to 255. Its **least significant bit** is the rightmost binary digit. Changing that bit changes the channel value by at most 1.

For example, `10010110` (150) becomes `10010111` (151) if its LSB is set to 1.

### Worked example: set and read one bit

In [ ]:
def set_LSB(value, bit):
    """Return an integer with its least significant bit set to 0 or 1."""
    if bit not in ("0", "1"):
        raise ValueError("bit must be '0' or '1'")
    return (value & 254) if bit == "0" else (value | 1)

def get_LSB(value):
    return str(value & 1)

value = 150
for bit in ("0", "1"):
    changed = set_LSB(value, bit)
    print(value, format(value, "08b"), "→", changed, format(changed, "08b"), "LSB", get_LSB(changed))

assert set_LSB(150, "1") == 151
assert get_LSB(151) == "1"

### Your turn — small modifications
1. Change the starting value to 201 and test setting its LSB to `0` and `1`.
2. Explain why `set_LSB(201, "1")` does not change the value.
3. What is the **maximum numerical difference** between an original channel and its LSB-modified value?

In [ ]:
# TODO: Run two examples with value = 201 and print binary before/after.
value = 201
# your code here

### From letters to bits
Python stores text as bytes when we encode it. For this lab, the required payload contains only ASCII characters, so each character occupies one byte. We use UTF-8 encoding and an 8-bit zero terminator to mark the end of the message.

In [ ]:
sample = "HI"
raw = sample.encode("utf-8")
print("Characters:", list(sample))
print("Byte values:", list(raw))
print("Bits:", " ".join(format(b, "08b") for b in raw))
print("Terminator:", format(0, "08b"))

## Part 2 — Calculate capacity before writing 
The supplied `steganography_LSB.py` uses **all four RGBA channels** and stores **8 bits across two pixels** (4 channels × 2 pixels). It appends one zero byte as an end marker.

**Important correction to the original PDF:** The requested string `TARGET:192.168.1.50` cannot fit in the *first 10 RGBA pixels*. It requires **20 ASCII bytes + 1 terminator = 21 bytes = 42 RGBA pixels** using the supplied algorithm. In this notebook, use the **first 42 pixels** (or more, if you change the payload).

**Transparency caveat:** The supplied implementation also modifies alpha-channel LSBs. On some images, that can slightly change transparency. We will preserve its behavior for compatibility, then discuss an RGB-only alternative.

### Worked capacity calculation
- One RGBA pixel provides 4 LSBs (4 bits).
- Two RGBA pixels provide 8 bits (1 byte).
- Reserve one byte for the zero terminator.
- Maximum ASCII message bytes in an RGBA image = `floor(width × height × 4 / 8) − 1`.

**Question:** How many message bytes fit in a 100 × 100 RGBA image using this exact method? How would an RGB-only, one-LSB-per-channel method differ?

In [ ]:
def max_message_bytes(width, height, channels=4, terminator_bytes=1):
    capacity_bytes = (width * height * channels) // 8
    return max(0, capacity_bytes - terminator_bytes)

print("100×100 RGBA:", max_message_bytes(100, 100, 4), "message bytes")
print("100×100 RGB:", max_message_bytes(100, 100, 3), "message bytes")

payload = "TARGET:192.168.1.50"
required_bytes = len(payload.encode("utf-8")) + 1
print("Payload + terminator:", required_bytes, "bytes")
print("Pixels needed (supplied RGBA method):", required_bytes * 2)
assert required_bytes == 21

## Part 3 — Manual LSB encoding and extraction 
Read `Lab8/steganography_LSB.py` and locate these functions: `set_LSB`, `get_LSB`, `get_pixel_pairs`, `hide_message`, `extract_message`.

**Scaffolding plan:** First test the supplied functions, then build your own small wrapper with input validation. Finally implement a decoder using the pseudocode below. Do not simply paste the entire reference script into the notebook as your final answer.

**Algorithm:** Convert carrier to RGBA → convert message to UTF-8 bytes → append `0x00` → for each byte, write its eight bits into the RGBA channels of two successive pixels → save PNG. Decryption reverses those steps and stops at the zero terminator.

In [ ]:
# Guided reference test: import the supplied script without modifying it.
import sys
sys.path.insert(0, str(ASSET_DIR.resolve()))
from steganography_LSB import hide_message, extract_message

TEST_STEGO = OUTPUT_DIR / "python_test_stego.png"
hide_message(str(PYTHON_IMG), "HI", str(TEST_STEGO))
recovered = extract_message(str(TEST_STEGO))
print("Recovered:", recovered)
assert recovered == "HI"

### Task 3A — Inspect the reference code 
Answer in Markdown:
1. Why does the reference code call `.convert("RGBA")`?
2. Why are two pixels used per byte?
3. What does `message += chr(0)` accomplish?
4. What happens if the payload exceeds image capacity? How could your code detect this *before* editing any pixels?
5. Why is PNG preferable to JPEG for preserving exact LSB values?

### Task 3B — Implement a validated encoder (partial starter)
Use the supplied `hide_message` function internally, but add **your own** capacity and input checks. The message is the fictional label `TARGET:192.168.1.50`.

**Your code must:** verify the source exists; reject an empty message; calculate required pixels; check capacity; write `company_logo_stego.png`; print the output path.

In [ ]:
def encode_checked(source_path, message, output_path):
    """Validate a short ASCII message, then call the supplied encoder."""
    source_path = Path(source_path)
    output_path = Path(output_path)
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
    if not message:
        raise ValueError("Message must not be empty")
    if "\x00" in message:
        raise ValueError("Zero byte is reserved as a terminator")
    with Image.open(source_path) as image:
        available_pixels = image.width * image.height
    needed_pixels = 2 * (len(message.encode("utf-8")) + 1)
    # TODO 1: If needed_pixels exceeds available_pixels, raise ValueError.
    # TODO 2: Call hide_message(...) with the three parameters.
    # TODO 3: Return output_path.
    raise NotImplementedError("Complete TODOs before running")

# After implementing, uncomment:
# LOGO_STEGO = encode_checked(LOGO, "TARGET:192.168.1.50", OUTPUT_DIR / "company_logo_stego.png")
# print(LOGO_STEGO)

### Task 3C — Build a decoder 
Start with the code below. Complete the missing steps yourself. Your decoder should **not** call the supplied `extract_message` function.

Pseudocode:
1. Open PNG and convert to RGBA.
2. Read pixels in pairs.
3. Collect the four LSBs from the first pixel and four from the second.
4. Convert the eight bits into one byte.
5. If byte is 0, stop; otherwise append it.
6. Decode collected bytes as UTF-8 text.

**Tip:** `int("01001000", 2)` converts a binary string to a decimal integer.

In [ ]:
def decode_manual(path):
    with Image.open(path) as image:
        pixels = list(image.convert("RGBA").getdata())
    decoded_bytes = bytearray()
    for index in range(0, len(pixels) - 1, 2):
        first, second = pixels[index], pixels[index + 1]
        # TODO: collect the eight LSBs from first + second
        # bits = ...
        # TODO: convert the 8-bit string to an integer
        # value = ...
        # TODO: stop if value == 0; otherwise append to decoded_bytes
        pass
    # TODO: return decoded_bytes.decode("utf-8")
    raise NotImplementedError("Complete the decoder")

# Test after completing both functions:
# assert decode_manual(LOGO_STEGO) == "TARGET:192.168.1.50"
# assert extract_message(str(LOGO_STEGO)) == "TARGET:192.168.1.50"

## Part 4 — Stepic: second implementation 
Stepic is a third-party library that hides bytes in image pixels. Use it to encode the **fictional** port list `[80, 443, 3478, 5060]` in `profile.png`, then recover it.

**Note:** Stepic typically works with RGB image data. Convert the carrier to RGB before encoding; save the result as PNG. Library behavior may vary by environment—document any installation or compatibility issue rather than inventing an output.

### Guided example: create a byte payload

In [ ]:
ports = [80, 443, 3478, 5060]
port_text = ",".join(str(port) for port in ports)
port_bytes = port_text.encode("utf-8")
print("Message:", port_text)
print("Bytes:", port_bytes)
assert port_text == "80,443,3478,5060"

### Task 4A — Encode and save
Use `stepic.encode(image, payload_bytes)` and `.save(...)`. The starter below supplies the workflow but leaves the central operation to you.

In [ ]:
# If Stepic is installed:
# import stepic
# with Image.open(PROFILE) as source:
#     rgb_image = source.convert("RGB")
#     # TODO: call stepic.encode(rgb_image, port_bytes)
#     # TODO: save returned image to OUTPUT_DIR / "profile_secret.png"
#     pass

### Task 4B — Decode and verify
Use `stepic.decode(image)` and compare the result with `port_text`. Some versions return text directly; if yours returns bytes, decode them with UTF-8. Record the exact recovered value.

**Analysis:** Why is it important to verify extraction rather than assuming a successful save means the payload survived?

In [ ]:
# TODO: Open profile_secret.png, call stepic.decode, normalize to text if needed.
# TODO: assert recovered_port_text == port_text
# Print the recovered text.

## Part 5 — File integrity, visual, and pixel analysis 

In Part 0 you recorded **baseline SHA-256 hashes**. Now use those values to investigate your outputs. 

1. Compare the **original file hash** with the **stego-image hash** after embedding.
2. Extract the hidden message, then hash the **same stego-image** again.
3. Compare the stego-image hash **before and after extraction**.
4. Compare file sizes and decoded pixel values, and visually inspect the images.

### Worked comparison function

In [ ]:
def compare_images(original_path, stego_path):
    with Image.open(original_path) as original, Image.open(stego_path) as stego:
        a = list(original.convert("RGBA").getdata())
        b = list(stego.convert("RGBA").getdata())
        if original.size != stego.size:
            raise ValueError("Image dimensions differ")
    changed_pixels = sum(p != q for p, q in zip(a, b))
    changed_channels = sum(x != y for p, q in zip(a, b) for x, y in zip(p, q))
    return {
        "original_bytes": Path(original_path).stat().st_size,
        "stego_bytes": Path(stego_path).stat().st_size,
        "original_sha256": sha256_file(original_path),
        "stego_sha256": sha256_file(stego_path),
        "same_file_bytes": sha256_file(original_path) == sha256_file(stego_path),
        "changed_pixels": changed_pixels,
        "changed_channels": changed_channels,
        "total_pixels": len(a),
    }

print("Reference test:", compare_images(PYTHON_IMG, TEST_STEGO))


### Task 5A — Reuse your baseline hashes 

After completing Parts 3 and 4, compare the original and stego-images. Use the starter code below to:

1. Calculate the SHA-256 hash of each stego-image.
2. Compare it against `original_logo_hash` or `original_profile_hash` from Part 0.
3. Record the file-size difference and number/percentage of changed pixels.
4. Explain **what the hash comparison establishes and what it cannot establish**.

**Checkpoint:** If two images look identical but their hashes differ, does that prove one contains a hidden message? Why or why not? What additional evidence would you examine?

In [ ]:
# Task 5A — Complete after Parts 3 and 4.
# Uncomment and adapt these lines; keep the earlier baseline variables.
#
# logo_stats = compare_images(LOGO, LOGO_STEGO)
# profile_stego = OUTPUT_DIR / "profile_secret.png"
# profile_stats = compare_images(PROFILE, profile_stego)
#
# print("Original logo baseline matches current original:",
#       original_logo_hash == sha256_file(LOGO))
# print("Original logo vs. stego identical:",
#       original_logo_hash == sha256_file(LOGO_STEGO))
# print("Original profile vs. stego identical:",
#       original_profile_hash == sha256_file(profile_stego))
#
# print("Logo pixel comparison:", logo_stats)
# print("Profile pixel comparison:", profile_stats)
#
# TODO: Calculate and print the percentage of changed pixels for each pair.
# TODO: In a Markdown cell, explain what your actual hash results demonstrate.


### Task 5B — Verify extraction did not modify the stego-image

A decoder should **read** the image without rewriting it. Demonstrate this by hashing a stego-image before and after extracting its hidden message.

**Guided example:** The manual decoder is shown below. 

**Your turn:** Repeat the same test for the Stepic image and its decoder.

A matching *before/after* hash supports the conclusion that the extraction operation did not change the file's bytes. It does not prove the extracted message is trustworthy; verify the recovered text separately.

In [ ]:
# Run after completing your manual decoder in Part 3.
# before_extract = sha256_file(LOGO_STEGO)
# recovered = decode_manual(LOGO_STEGO)
# after_extract = sha256_file(LOGO_STEGO)
#
# print("Recovered message:", recovered)
# print("Before extraction:", before_extract)
# print("After extraction: ", after_extract)
# assert before_extract == after_extract, "Extraction unexpectedly changed the file!"
# assert recovered == "TARGET:192.168.1.50"
#
# TODO: Repeat for profile_secret.png with stepic.decode(...).
# TODO: Record whether the hash changed and explain why this matters.


### Task 5C — View the original and modified images
Use the display helper below to compare the images. Record an observation; avoid claiming 'no change' based solely on appearance.

In [ ]:
from IPython.display import display

def show_pair(original_path, stego_path):
    with Image.open(original_path) as original, Image.open(stego_path) as stego:
        print("Original:")
        display(original.copy().resize((300, 300)))
        print("Stego:")
        display(stego.copy().resize((300, 300)))

# TODO: show_pair(LOGO, LOGO_STEGO)
# TODO: show_pair(PROFILE, OUTPUT_DIR / "profile_secret.png")

## Part 6 — Independent security analysis and reflection 
Write **250–350 words**, using your actual results, addressing:
1. How is LSB steganography different from encryption? Can a hidden message still be read by someone who knows the extraction method?
2. What evidence shows that your two payloads were correctly recovered?
3. Why does image capacity depend on dimensions, channels, and terminator/length encoding?
4. Explain why PNG is more appropriate than JPEG for this exact LSB approach.
5. **Use your SHA-256 results:** What changed between each original and stego-image? Did extraction alter either stego-image? Why do different hashes **not** prove steganography?
6. Identify **two limitations** of these techniques and **two defensive investigation steps** that could be used in an authorized forensic review.
7. Explain why the fictional `TARGET` label and port list are **not evidence of a real scan or data theft**.

**Report quality:** Show code and relevant outputs, then explain *what each output demonstrates*. Do not submit unexplained screenshots.

### Reflection response
_Write your 250–350-word analysis here._

## Submission checklist
- [ ] Environment and evidence inventory completed; original filenames and baseline hashes recorded, then reused in Part 5.
- [ ] LSB binary experiments and capacity calculations explained.
- [ ] Validated manual encoder and independently written decoder completed.
- [ ] `company_logo_stego.png` created and payload recovered exactly.
- [ ] Stepic `profile_secret.png` created and port list recovered exactly (or documented reproducible compatibility issue).
- [ ] Original/stego SHA-256, visual, size, and pixel comparisons completed; extraction integrity verified.
- [ ] Two robustness tests documented.
- [ ] 250–350-word analysis completed, with evidence and limitations.
- [ ] PDF exported with filename `Lab8-FirstName-LastName-StudentNumber.pdf`.

## Grading rubric (100 marks)
| Criterion | Marks |
|---|---:|
| Part 0–1: setup, binary and LSB understanding | 15 |
| Part 2: accurate capacity calculations and reasoning | 10 |
| Part 3: manual encoder/decoder, validation and testing | 30 |
| Part 4: Stepic encode/decode and verification | 15 |
| Part 5: SHA-256 baseline comparison, extraction integrity, pixel analysis and interpretation | 15 |
| Part 6: security analysis, reflection and report clarity | 15 |
| **Total** | **100** |

**Optional extension (not required for full marks):** Compare the supplied RGBA algorithm with an RGB-only design and explain how avoiding alpha modifications changes capacity and image integrity.